# Daily Challenge: Classification with Neural Networks in TensorFlow

## 1. Understanding Classification Types

**Binary Classification**

Binary classification involves predicting one of exactly two possible classes. The output is typically 0 or 1, and the model uses a sigmoid activation in the final layer to produce a probability.

*Example*: Predicting whether an email is spam (1) or not spam (0).

---

**Multi-class Classification**

Multi-class classification involves predicting exactly one class out of three or more mutually exclusive classes. The model uses a softmax activation in the final layer, producing a probability distribution across all classes that sums to 1.

*Example*: Classifying an image of a handwritten digit as one of 0-9 (10 classes, only one is correct).

---

**Multi-label Classification**

Multi-label classification involves predicting multiple labels simultaneously for a single input, where labels are not mutually exclusive. The model uses a sigmoid activation independently on each output node, since each label is treated as its own binary decision.

*Example*: Tagging a news article with multiple relevant topics at once, such as "politics", "economy", and "international" all applying to the same article.

## 2. Setup and Dataset Creation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
tf.random.set_seed(42)

In [ ]:
samples = 1000
X, y = make_circles(samples,
                     noise=0.03,
                     random_state=42)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFirst 5 rows of X:\n", X[:5])
print("\nFirst 5 values of y:\n", y[:5])

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=15)
plt.title("make_circles Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.colorbar(label="Class")
plt.tight_layout()
plt.show()

## 3. Build a Basic Neural Network Model

In [ ]:
model_basic = tf.keras.Sequential([
    tf.keras.layers.Dense(1)
])

model_basic.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.SGD(),
    metrics=["accuracy"]
)

history_basic = model_basic.fit(X, y, epochs=20, verbose=0)

loss_basic, acc_basic = model_basic.evaluate(X, y, verbose=0)
print(f"Basic Model — Loss: {loss_basic:.4f} | Accuracy: {acc_basic:.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_basic.history["loss"], label="Loss")
plt.plot(history_basic.history["accuracy"], label="Accuracy")
plt.title("Basic Model (1 Dense layer, SGD) — Training History")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.show()

The basic model has only a single dense layer with no activation function or hidden layers, so it can only learn a linear decision boundary. Since `make_circles` produces a non-linearly separable dataset (one class forms a ring around the other), this model is expected to perform poorly — accuracy should hover near 50%, equivalent to random guessing.

## 4. Improve the Model

In [ ]:
model_improved = tf.keras.Sequential([
    tf.keras.layers.Dense(4),
    tf.keras.layers.Dense(4),
    tf.keras.layers.Dense(1)
])

model_improved.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"]
)

history_improved = model_improved.fit(X, y, epochs=100, verbose=0)

loss_improved, acc_improved = model_improved.evaluate(X, y, verbose=0)
print(f"Improved Model — Loss: {loss_improved:.4f} | Accuracy: {acc_improved:.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_improved.history["loss"], label="Loss")
plt.plot(history_improved.history["accuracy"], label="Accuracy")
plt.title("Improved Model (more layers, Adam, 100 epochs) — Training History")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.show()

Adding more layers, more neurons, more training epochs, and switching from SGD to Adam (which adapts the learning rate automatically) improves convergence speed and accuracy. However, without non-linear activation functions, stacking purely linear Dense layers is still mathematically equivalent to a single linear transformation, so the model may still struggle to fully separate the circular data.

## 5. Visualize the Decision Boundary

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """
    Plots the decision boundary created by a trained model on 2D data.
    """
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    preds = model.predict(grid, verbose=0)

    if preds.shape[-1] == 1:
        preds = (preds > 0.5).astype(int).reshape(xx.shape)
    else:
        preds = np.argmax(preds, axis=1).reshape(xx.shape)

    plt.figure(figsize=(6, 6))
    plt.contourf(xx, yy, preds, alpha=0.4, cmap="coolwarm")
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm",
                edgecolor="black", s=15)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.tight_layout()
    plt.show()


print("Function plot_decision_boundary defined.")

In [ ]:
plot_decision_boundary(model_basic, X, y, title="Basic Model — Decision Boundary")
plot_decision_boundary(model_improved, X, y, title="Improved Model (no activation) — Decision Boundary")

Both decision boundaries are straight lines (or close to it), confirming that without non-linear activation functions, the network cannot learn the circular pattern, regardless of depth.

## 6. Incorporate Activation Functions

In [ ]:
model_activated = tf.keras.Sequential([
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_activated.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=["accuracy"]
)

history_activated = model_activated.fit(X, y, epochs=100, verbose=0)

loss_activated, acc_activated = model_activated.evaluate(X, y, verbose=0)
print(f"Activated Model (ReLU + Sigmoid) — Loss: {loss_activated:.4f} | Accuracy: {acc_activated:.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_activated.history["loss"], label="Loss")
plt.plot(history_activated.history["accuracy"], label="Accuracy")
plt.title("Model with ReLU + Sigmoid — Training History")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.show()

plot_decision_boundary(model_activated, X, y, title="Model with Activation Functions — Decision Boundary")

Adding ReLU activations in the hidden layers introduces non-linearity, allowing the network to bend its decision boundary into a curved shape that can separate the two concentric circles. The sigmoid activation on the output layer squashes predictions into a [0, 1] probability range, suitable for binary classification with `BinaryCrossentropy` loss. This model is expected to achieve much higher accuracy than the previous two.

## 7. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")

In [ ]:
model_final = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_final.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=["accuracy"]
)

history_final = model_final.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    verbose=0
)

print("Training complete.")

## 8. Evaluate and Visualize Final Model Performance

In [ ]:
train_loss, train_acc = model_final.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc   = model_final.evaluate(X_test, y_test, verbose=0)

print("Final Model Performance")
print("-" * 40)
print(f"Training — Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")
print(f"Test     — Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_final.history["loss"], label="Train Loss")
axes[0].plot(history_final.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_final.history["accuracy"], label="Train Accuracy")
axes[1].plot(history_final.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
plot_decision_boundary(model_final, X_train, y_train, title="Final Model — Training Data")
plot_decision_boundary(model_final, X_test, y_test, title="Final Model — Test Data")

**Comparison of all models**

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Basic (1 layer, SGD, no activation)",       "Accuracy": acc_basic},
    {"Model": "Improved (3 layers, Adam, no activation)",  "Accuracy": acc_improved},
    {"Model": "With ReLU + Sigmoid activations",           "Accuracy": acc_activated},
    {"Model": "Final model (test set)",                    "Accuracy": test_acc},
])

print(comparison.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.barh(comparison["Model"], comparison["Accuracy"], color="steelblue")
plt.xlim(0, 1)
plt.title("Model Accuracy Comparison")
plt.xlabel("Accuracy")
plt.tight_layout()
plt.show()

## 9. Key Takeaways

This exercise demonstrated the full workflow of building a neural network classifier on a non-linearly separable dataset (`make_circles`). The single-layer model without activation functions performed close to random guessing, since it can only express linear decision boundaries. Adding more layers and neurons alone did not solve the problem, because stacking linear transformations without non-linear activations is still mathematically linear. The key breakthrough came from introducing ReLU activations in the hidden layers and a sigmoid activation in the output layer — this allowed the network to learn a curved, non-linear decision boundary that correctly separates the two concentric circles.

Switching the optimizer from SGD to Adam also accelerated convergence, since Adam adapts the learning rate per parameter rather than using a single fixed rate.

Visualizing the data distribution before modeling was essential: it immediately revealed that the classes were not linearly separable, which explained why the early models failed and motivated the need for activation functions. Likewise, plotting the decision boundary at each stage made the model's limitations and improvements visually obvious, complementing the numeric accuracy and loss metrics.

Overall, this exercise highlights that improving a neural network is not just about adding more layers or training longer — non-linear activation functions are what give neural networks their expressive power, and tuning the optimizer and learning rate can significantly affect how efficiently the network learns.